# Hamiltonian Neural Networks: Physics as Inductive Bias

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/hamiltonian_neural_networks.ipynb)

A plain neural network trained to predict a physical system's next state slowly
leaks energy: its trajectories spiral in or out. A **Hamiltonian Neural Network**
(Greydanus, Dzamba & Yosinski, NeurIPS 2019) instead learns a scalar energy
`H(q, p)` and reads the dynamics off its derivatives through Hamilton's equations,
so energy conservation is built into the architecture.

This notebook builds one from scratch in PyTorch, compares it against an ordinary
MLP on the ideal mass-spring, the pendulum, and a two-body orbit, and shows the
difference: the MLP drifts, the HNN conserves.

Companion post: https://sesen.ai/blog/hamiltonian-neural-networks

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)

## The mass-spring and its dynamics

An ideal mass on a spring has one conserved quantity, its total energy
`H(q, p) = (q^2 + p^2) / 2`. Hamilton's equations give the velocity field, and we
integrate it with fourth-order Runge-Kutta to draw trajectories.

In [ ]:
def spring_field(s):                       # s = [q, p]
    q, p = s[..., 0], s[..., 1]
    return np.stack([p, -q], axis=-1)      # (dq/dt, dp/dt)

def spring_energy(s):
    return 0.5 * (s[..., 0] ** 2 + s[..., 1] ** 2)

def rk4(f, s, dt):                         # one integrator step
    k1 = f(s); k2 = f(s + .5*dt*k1); k3 = f(s + .5*dt*k2); k4 = f(s + dt*k3)
    return s + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

def trajectory(f, s0, dt, n):
    out = np.zeros((n, len(s0))); s = s0.astype(float)
    for i in range(n):
        out[i] = s; s = rk4(f, s, dt)
    return out

## Training data

The network never sees an energy value. It sees states `(q, p)` and their
velocities `(dq/dt, dp/dt)`, sampled along 40 short orbits at random radii.

In [ ]:
def sample_orbits(field, radii, seed):
    rng = np.random.default_rng(seed)
    orbits = [trajectory(field, np.array([r*np.cos(a), r*np.sin(a)]), 0.1, 50)
              for r, a in zip(rng.uniform(*radii, 40), rng.uniform(0, 2*np.pi, 40))]
    X = np.concatenate(orbits).astype(np.float32)
    Y = (field(X) + rng.normal(scale=0.01, size=X.shape)).astype(np.float32)
    return X, Y

Xs, Ys = sample_orbits(spring_field, (1.0, 2.5), seed=1)
print(Xs.shape, Ys.shape)

## The baseline MLP and the HNN

The **baseline** maps a state straight to a velocity. The **HNN** maps the state to
a single number `H`, then produces the velocity as the *symplectic gradient*:
differentiate `H` with autograd, swap the two components, and flip one sign. That
is exactly Hamilton's equations `dq/dt = dH/dp`, `dp/dt = -dH/dq`.

In [ ]:
def baseline(nin, h=200):
    return nn.Sequential(nn.Linear(nin, h), nn.Tanh(),
                         nn.Linear(h, h), nn.Tanh(), nn.Linear(h, nin))

class HNN(nn.Module):
    def __init__(self, nin, h=200):
        super().__init__(); self.d = nin // 2
        self.H = nn.Sequential(nn.Linear(nin, h), nn.Tanh(),
                               nn.Linear(h, h), nn.Tanh(), nn.Linear(h, 1))
    def forward(self, x):
        x = x.requires_grad_(True)
        dH = torch.autograd.grad(self.H(x).sum(), x, create_graph=True)[0]
        return torch.cat([dH[:, self.d:], -dH[:, :self.d]], dim=1)

def train(model, X, Y, steps=3000):
    Xt, Yt = torch.tensor(X), torch.tensor(Y)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=1000, gamma=0.3)
    losses = []
    for _ in range(steps):
        opt.zero_grad(); loss = ((model(Xt) - Yt) ** 2).mean()
        loss.backward(); opt.step(); sched.step(); losses.append(loss.item())
    return losses

def learned_field(model):
    def f(s):
        d = model(torch.tensor(np.atleast_2d(s), dtype=torch.float32))
        return d.detach().numpy().reshape(np.shape(s))
    return f

## Train both models and roll them out

Both fit the vector field to about the same loss. The difference shows up only when
we integrate the learned dynamics forward for 80 orbits and track the energy.

In [ ]:
torch.manual_seed(0)
hnn_s = HNN(2); mlp_s = baseline(2)          # build HNN first
lh_s = train(hnn_s, Xs, Ys)
lm_s = train(mlp_s, Xs, Ys)
print(f"final loss  MLP={lm_s[-1]:.5f}  HNN={lh_s[-1]:.5f}")

DT, ROLL = 0.1, 5000
s0 = np.array([2.3, 0.0])
true_s = trajectory(spring_field, s0, DT, ROLL)
hr_s = trajectory(learned_field(hnn_s), s0, DT, ROLL)
mr_s = trajectory(learned_field(mlp_s), s0, DT, ROLL)
for name, r in [("MLP", mr_s), ("HNN", hr_s)]:
    e = spring_energy(r)
    print(f"{name}: energy drift over 80 orbits = {100*(e[-1]-e[0])/e[0]:+.1f}%")

## Phase-space rollout

The baseline spirals inward as energy drains; the HNN retraces the true circle.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
for a, r, name in [(ax[0], mr_s, "Baseline MLP"), (ax[1], hr_s, "Hamiltonian NN")]:
    a.plot(true_s[:, 0], true_s[:, 1], "k--", alpha=.5, label="true orbit")
    a.plot(r[:, 0], r[:, 1], lw=.8, label="rollout")
    a.set_aspect("equal"); a.set_xlabel("q"); a.set_ylabel("p")
    a.set_title(name); a.legend()
plt.tight_layout(); plt.show()

## Energy over time

In [ ]:
t = np.arange(ROLL) * DT
plt.figure(figsize=(9, 4))
plt.plot(t, spring_energy(true_s), "k--", label="true")
plt.plot(t, spring_energy(hr_s), label="HNN")
plt.plot(t, spring_energy(mr_s), label="MLP")
plt.xlabel("time"); plt.ylabel("total energy"); plt.legend(); plt.show()

## Training loss

Both models converge to about the same vector-field MSE. Fit quality is not the
difference between them.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(lm_s, label="MLP"); plt.plot(lh_s, label="HNN")
plt.yscale("log"); plt.xlabel("training step"); plt.ylabel("MSE"); plt.legend(); plt.show()

## The learned Hamiltonian

The HNN was trained only on velocities, but because its output is the gradient of a
scalar, that scalar exists. Plot it and you recover the true energy surface up to a
constant.

In [ ]:
gx, gy = np.meshgrid(np.linspace(-2.6, 2.6, 100), np.linspace(-2.6, 2.6, 100))
grid = torch.tensor(np.stack([gx, gy], -1).reshape(-1, 2).astype(np.float32))
with torch.no_grad():
    Hn = hnn_s.H(grid).numpy().reshape(gx.shape)
Hn -= Hn.min()
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].contourf(gx, gy, spring_energy(np.stack([gx, gy], -1)), levels=18)
ax[0].set_title("true H"); ax[0].set_aspect("equal")
ax[1].contourf(gx, gy, Hn, levels=18)
ax[1].set_title("learned H"); ax[1].set_aspect("equal")
plt.tight_layout(); plt.show()

## A nonlinear system: the pendulum

The `sin q` restoring force makes the pendulum nonlinear. Retrain both models
unchanged and the contrast survives.

In [ ]:
def pend_field(s):
    q, p = s[..., 0], s[..., 1]
    return np.stack([p, -np.sin(q)], axis=-1)

def pend_energy(s):
    q, p = s[..., 0], s[..., 1]
    return 0.5 * p ** 2 + (1.0 - np.cos(q))

Xp, Yp = sample_orbits(pend_field, (1.3, 2.3), seed=3)
torch.manual_seed(0)
hnn_p = HNN(2); mlp_p = baseline(2)
train(hnn_p, Xp, Yp); train(mlp_p, Xp, Yp)

x0 = np.array([2.1, 0.0]); NP = 1500
tp = trajectory(pend_field, x0, DT, NP)
hp = trajectory(learned_field(hnn_p), x0, DT, NP)
mp = trajectory(learned_field(mlp_p), x0, DT, NP)
for name, r in [("MLP", mp), ("HNN", hp)]:
    e = pend_energy(r)
    print(f"{name}: pendulum energy drift = {100*(e[-1]-e[0])/e[0]:+.1f}%")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].plot(tp[:, 0], tp[:, 1], "k--", alpha=.5); ax[0].plot(mp[:, 0], mp[:, 1], lw=.7, label="MLP")
ax[0].plot(hp[:, 0], hp[:, 1], lw=.9, label="HNN"); ax[0].set_title("phase space")
ax[0].set_aspect("equal"); ax[0].legend()
tt = np.arange(NP) * DT
ax[1].plot(tt, pend_energy(tp), "k--"); ax[1].plot(tt, pend_energy(hp), label="HNN")
ax[1].plot(tt, pend_energy(mp), label="MLP"); ax[1].set_title("energy"); ax[1].legend()
plt.tight_layout(); plt.show()

## Two-body capstone: a planet orbiting a star

A 4-D system: state `(x, y, px, py)`, energy
`H = (px^2 + py^2)/2 - 1/sqrt(x^2 + y^2)`. The HNN keeps a bound orbit; the baseline
lets the planet spiral off.

In [ ]:
EPS = 0.1
def kepler_field(s):
    x, y, px, py = s[..., 0], s[..., 1], s[..., 2], s[..., 3]
    r3 = (x*x + y*y + EPS*EPS) ** 1.5
    return np.stack([px, py, -x/r3, -y/r3], axis=-1)

def kepler_energy(s):
    x, y, px, py = s[..., 0], s[..., 1], s[..., 2], s[..., 3]
    return 0.5*(px*px + py*py) - 1.0/np.sqrt(x*x + y*y + EPS*EPS)

def kepler_data(seed):
    rng = np.random.default_rng(seed); S = []
    for _ in range(50):
        r0 = rng.uniform(0.9, 1.6); vc = 1/np.sqrt(r0); v = vc*rng.uniform(0.85, 1.1)
        a = rng.uniform(0, 2*np.pi); c, s = np.cos(a), np.sin(a)
        S.append(trajectory(kepler_field, np.array([r0*c, r0*s, -v*s, v*c]), 0.05, 60))
    X = np.concatenate(S); Y = kepler_field(X) + rng.normal(scale=0.01, size=(len(X), 4))
    return X.astype(np.float32), Y.astype(np.float32)

Xk, Yk = kepler_data(1)
torch.manual_seed(0)
hnn_k = HNN(4); mlp_k = baseline(4)
train(hnn_k, Xk, Yk); train(mlp_k, Xk, Yk)

r0 = 1.2; v = 0.95/np.sqrt(r0); s0k = np.array([r0, 0.0, 0.0, v]); NK = 3000
tk = trajectory(kepler_field, s0k, 0.05, NK)
hk = trajectory(learned_field(hnn_k), s0k, 0.05, NK)
mk = trajectory(learned_field(mlp_k), s0k, 0.05, NK)
print("HNN energy band = {:.1f}%".format(
    100*(kepler_energy(hk).max()-kepler_energy(hk).min())/abs(kepler_energy(hk)[0])))

fig, ax = plt.subplots(1, 2, figsize=(11, 5.2))
for a, tr, name in [(ax[0], mk, "Baseline MLP"), (ax[1], hk, "Hamiltonian NN")]:
    a.plot(tk[:, 0], tk[:, 1], "k--", alpha=.5)
    a.plot(tr[:, 0], tr[:, 1], lw=.7)
    a.scatter([0], [0], marker="*", s=180, color="orange")
    a.set_xlim(-1.9, 1.9); a.set_ylim(-1.9, 1.9); a.set_aspect("equal"); a.set_title(name)
plt.tight_layout(); plt.show()

## Exercises

1. **Push the pendulum to the separatrix.** Start it near `q = pi` (energy close to 2)
   so it almost goes over the top. Does the HNN still conserve energy? What does the
   baseline do?
2. **Add friction.** Change `spring_field` to `dp/dt = -q - 0.05*p` (a damped spring)
   and retrain both. Now the MLP fits the decaying orbit better than the HNN. Explain
   why the conservation prior hurts here.
3. **Symplectic integrator.** Replace `rk4` with a leapfrog integrator for the
   rollout. Does the HNN's small energy wobble shrink?
4. **Activation matters.** Swap `nn.Tanh()` for `nn.ReLU()` in both networks and
   re-run. Watch the HNN's learned vector field turn jagged. Why does the HNN need a
   smooth activation more than the baseline does?